In [ ]:
import warnings
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import nltk
import pickle

from collections import Counter
from datasets import load_dataset
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from tqdm import tqdm

# Enable tqdm integration with pandas
tqdm.pandas()

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(pkg, quiet=True)

warnings.filterwarnings('ignore')

In [ ]:
# Load pickle file from preprocessing.ipynb
with open('case_data.pkl', 'rb') as file:
    df = pickle.load(file)

# Print first five rows
df.head()

## Generate Summaries

In [ ]:
# Load T5-small model pipeline
t5_pipe = pipeline(
    "summarization",
    model="t5-small",
    tokenizer="t5-small",
    framework="pt",
    device=-1
)

# Split text into word-boundary chunks to bypass token limits
def chunk_text(text, chunk_size=1200):
    words = text.split()
    chunks, current, current_len = [], [], 0

    for word in words:
        current.append(word)
        current_len += len(word) + 1

        if current_len >= chunk_size:
            chunks.append(' '.join(current))
            current, current_len = [], 0

    if current:
        chunks.append(' '.join(current))

    return chunks

# Run one T5 inference pass
def t5_summarize_chunk(text, max_length=80, min_length=15):
    result = t5_pipe(
        f"summarize: {text}",
        max_length=max_length,
        min_length=min_length,
        do_sample=False,
        truncation=True
    )

    return result[0]['summary_text']

# Summarize each chunk of the source text + combined intermediate summaries
def generate_t5_summary(text, level='short', max_input_chars=8000, max_chunks=5):
    text = text[:max_input_chars]
    chunks = chunk_text(text, chunk_size=1200)[:max_chunks]

    # Combine intermediate summaries
    intermediates = [t5_summarize_chunk(c, max_length=80, min_length=15) for c in chunks]
    combined = ' '.join(intermediates)

    # Define size thresholds
    level_params = {
        'long': dict(max_length=400, min_length=100),
        'short': dict(max_length=150, min_length=40),
        'tiny': dict(max_length=50,  min_length=10),
    }

    # Return final summary at desired granularity
    return t5_summarize_chunk(combined, **level_params[level])

In [ ]:
# Subset DataFrame to training data
train_df = (
    df[(df['split'] == 'train')
       & df['summary_long'].notna()
       & df['summary_short'].notna()
       & df['summary_tiny'].notna()]
    .reset_index(drop=True)
)

# Generate T5 summaries for all cases
for level in ['long', 'short', 'tiny']:
    col = f'gen_{level}'

    train_df[col] = train_df['cleaned_text'].apply(
        lambda t: generate_t5_summary(t, level=level)
    )

# Print first five cases' T5-generated summaries
train_df[['id', 'gen_long', 'gen_short', 'gen_tiny']].head()

## Compare Generated & Reference Summaries

In [ ]:
# Load sentence transformer model
st_model = SentenceTransformer('all-MiniLM-L6-v2')

# Calculate cosine similarity between sentence transformer embeddings
def semantic_similarity(gen, ref):
    if not gen or not ref:
        return np.nan
    
    # Embed summaries
    embs = st_model.encode([gen, ref], convert_to_numpy=True)

    # Return cosine similarity scores
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])


levels = [
    ('Tiny',  'gen_tiny',  'summary_tiny'),
    ('Short', 'gen_short', 'summary_short'),
    ('Long',  'gen_long',  'summary_long'),
]

sim_scores = {}

# Store cosine similarity scores
for label, gen_col, ref_col in levels:
    scores = train_df.apply(
        lambda r: semantic_similarity(r[gen_col], r[ref_col]), axis=1
    )

    sim_scores[label] = scores.values

# Convert to DataFrame
sim_df = pd.DataFrame(sim_scores, index=train_df['id'])

# Print mean cosine similarity scores by summary level
print(sim_df.mean().round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
level_labels = [label for label, *_ in levels]
colors = ['#2196F3', '#4CAF50', '#FF9800']

# Create bar chart of mean similarity by level
means = sim_df.mean().values
axes[0].bar(level_labels, means, color=colors)
axes[0].set_ylim(0, 1)
axes[0].set_title('Mean Semantic Similarity by Summary Level', fontsize=12)
axes[0].set_ylabel('Cosine Similarity')
for i, v in enumerate(means):
    axes[0].text(i, v + 0.015, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[0].axhline(0.7, color='red', linestyle='--', alpha=0.4, label='0.7 threshold')
axes[0].legend(fontsize=9)

# Create corresponding box plot to visualize distribution
bp = axes[1].boxplot(
    [sim_df[col].dropna().values for col in level_labels],
    labels=level_labels,
    patch_artist=True,
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylim(0, 1)
axes[1].set_title('Cosine Similarity Distribution by Level', fontsize=12)
axes[1].set_ylabel('Cosine Similarity')
axes[1].axhline(0.7, color='red', linestyle='--', alpha=0.4, label='0.7 threshold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Do T5-generated summaries align with their expert-authored references?

**Tiny summaries** tend to score highest on semantic similarity. T5's second-pass compression forces it to surface key information, which often overlaps with what the expert chose to highlight in their summary. Since both are so short, the embedding space has less noise and similarity is easier to achieve.

**Short summaries** show moderate similarity. The expanded budget allows T5 to capture more details, but the expert-authored summary includes additional context that T5 can't reconstruct within its allotted character window.

**Long summaries** typically score lowest. Since our hierarchical approach truncates reference summaries to the first 8,000 characters across no more than five chunks, later details are omitted entirely. Lower similarity scores reflect this gap in coverage.